# Bangladesh and the Flying Geese hypothesis: panel EDA

This reproducible notebook evaluates whether Bangladesh's industrial structure, trade integration and investment trajectory are consistent with the Flying Geese (FG) hypothesis. Countries: Bangladesh, China, Indonesia, Japan, Korea, Malaysia, Philippines, Singapore and Viet Nam. Annual WDI data, 1976–2025 where available. Missing values remain missing; no interpolation or zero-filling.

FG concerns changing comparative advantage and production/export composition—not just a rising manufacturing share. Aggregate WDI indicators can assess broad consistency but cannot establish product-level baton passing.

**References:** Akamatsu (1962), *The Developing Economies* 1(S1), 3–25; Dowling & Cheang (2000), *Journal of Asian Economics* 11(4), 443–463; World Bank (2018), [The Future of Manufacturing-Led Development in East Asia](https://documents1.worldbank.org/curated/en/116191523612461208/pdf/125189-PUB-PUBLIC-docdate-4-11-18.pdf); [WDI manufacturing metadata](https://databank.worldbank.org/metadataglossary/world-development-indicators/series/NV.IND.MANF.ZS); [WDI API documentation](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation).

In [ ]:
import warnings, requests, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.stattools import adfuller, kpss
warnings.filterwarnings('ignore')
COUNTRIES={'BGD':'Bangladesh','CHN':'China','IDN':'Indonesia','JPN':'Japan','KOR':'Korea, Rep.','MYS':'Malaysia','PHL':'Philippines','SGP':'Singapore','VNM':'Viet Nam'}
IND={'NV.IND.MANF.ZS':'manufacturing_gdp','NV.IND.TOTL.ZS':'industry_gdp','NE.EXP.GNFS.ZS':'exports_gdp','NE.IMP.GNFS.ZS':'imports_gdp','NE.TRD.GNFS.ZS':'trade_gdp','BX.KLT.DINV.WD.GD.ZS':'fdi_gdp','NE.GDI.FTOT.ZS':'gfcf_gdp','NE.GDI.TOTL.ZS':'gross_capital_gdp','NV.IND.MANF.KD.ZG':'manufacturing_growth','NY.GDP.MKTP.KD.ZG':'gdp_growth','TX.VAL.TECH.MF.ZS':'hightech_exports_share','NE.EXP.GNFS.KD.ZG':'exports_growth','NV.SRV.TOTL.ZS':'services_gdp','NV.AGR.TOTL.ZS':'agriculture_gdp'}
def fetch(code):
 url=f'https://api.worldbank.org/v2/country/{";".join(COUNTRIES)}/indicator/{code}'
 r=requests.get(url,params={'date':'1976:2025','format':'json','per_page':20000},timeout=60); r.raise_for_status(); j=r.json()
 return pd.DataFrame([{'iso3':x['countryiso3code'],'year':int(x['date']),'variable':IND[code],'value':x['value']} for x in (j[1] or [])])
long=pd.concat([fetch(c) for c in IND],ignore_index=True)
long['country']=long.iso3.map(COUNTRIES)
panel=long.pivot_table(index=['iso3','country','year'],columns='variable',values='value',aggfunc='first').reset_index().sort_values(['iso3','year'])
panel['ntc']=(panel.exports_gdp-panel.imports_gdp)/(panel.exports_gdp+panel.imports_gdp)
panel['exports_imports_ratio']=panel.exports_gdp/panel.imports_gdp.where(panel.imports_gdp.ne(0))
print('Rows/countries/years:',len(panel),panel.country.nunique(),panel.year.min(),panel.year.max())
display(pd.DataFrame({'n_nonmissing':panel.notna().sum(),'coverage_pct':(100*panel.notna().mean()).round(1)}))
print('Duplicate country-year:',panel.duplicated(['iso3','year']).sum())

## 1. Coverage and trajectories

High-tech exports are sparse and measure composition, not domestic technology content. Read GDP shares alongside real growth: manufacturing's share can fall while real output rises.

In [ ]:
countries=list(COUNTRIES.values())
variables=[('manufacturing_gdp','Manufacturing value added (% GDP)'),('exports_gdp','Exports (% GDP)'),('imports_gdp','Imports (% GDP)'),('gfcf_gdp','Gross fixed capital formation (% GDP)'),('fdi_gdp','FDI net inflows (% GDP)'),('hightech_exports_share','High-tech exports (% manufactured exports)'),('manufacturing_growth','Manufacturing growth (%)')]
for v,title in variables:
 fig,ax=plt.subplots(figsize=(12,5)); sns.lineplot(data=panel,x='year',y=v,hue='country',ax=ax); ax.set_title(title); ax.grid(alpha=.25); plt.tight_layout(); plt.show()
panel['decade']=(panel.year//10)*10
display(panel.groupby(['country','decade'])[[v for v,_ in variables]].median().round(2).tail(40))

## 2. Common-window comparison and trends

Comparisons use years where both Bangladesh and the comparator have observations. Trend tests use Spearman's rho and correct across tests using Benjamini–Hochberg. These are descriptive trends, not causal evidence.

In [ ]:
rows=[]
for v,_ in variables:
 for comp in [x for x in countries if x!='Bangladesh']:
  a=panel.loc[panel.country.eq('Bangladesh'),['year',v]].dropna().rename(columns={v:'bd'})
  b=panel.loc[panel.country.eq(comp),['year',v]].dropna().rename(columns={v:'peer'})
  z=a.merge(b,on='year').sort_values('year')
  if len(z): rows.append({'variable':v,'peer':comp,'n_common':len(z),'start':int(z.year.min()),'end':int(z.year.max()),'BD_change':z.bd.iloc[-1]-z.bd.iloc[0],'peer_change':z.peer.iloc[-1]-z.peer.iloc[0]})
common=pd.DataFrame(rows); display(common.round(3))
tr=[]
for country in countries:
 for v,_ in variables:
  z=panel.loc[panel.country.eq(country),['year',v]].dropna()
  if len(z)>=10:
   rho,p=stats.spearmanr(z.year,z[v]); slope=np.polyfit(z.year,z[v],1)[0]
   tr.append({'country':country,'variable':v,'n':len(z),'linear_slope':slope,'spearman_rho':rho,'p_raw':p})
trends=pd.DataFrame(tr); trends['p_BH']=multipletests(trends.p_raw,method='fdr_bh')[1]; display(trends.sort_values('p_BH').round(4))

## 3. Candidate break scan

A segmented trend is tested at candidate years with at least 10 observations on each side. Report the best raw p-value and Bonferroni-adjusted p-value across candidate dates. Selected dates are exploratory, not confirmed turning points.

In [ ]:
def scan_break(z,v,min_side=10):
 z=z[['year',v]].dropna().sort_values('year').copy(); out=[]
 if len(z)<2*min_side+3:return pd.DataFrame()
 for yr in z.year.unique()[min_side-1:-min_side]:
  q=z.copy(); q['post']=(q.year>yr).astype(int); q['t']=q.year-q.year.min(); q['tpost']=q.t*q.post
  fit=smf.ols(f'Q("{v}") ~ t + post + tpost',data=q).fit(cov_type='HC3')
  try:
   w=fit.wald_test('post=0, tpost=0',scalar=True); out.append({'year':int(yr),'n_pre':sum(q.year<=yr),'n_post':sum(q.year>yr),'wald':float(np.asarray(w.statistic).squeeze()),'p_raw':float(w.pvalue)})
  except Exception: pass
 r=pd.DataFrame(out)
 if len(r): r['p_bonf']=multipletests(r.p_raw,method='bonferroni')[1]
 return r
br=[]
for c in countries:
 for v in ['manufacturing_gdp','exports_gdp','imports_gdp','gfcf_gdp','fdi_gdp']:
  r=scan_break(panel[panel.country.eq(c)],v)
  if len(r):
   x=r.loc[r.p_raw.idxmin()].to_dict(); x.update(country=c,variable=v,n_total=panel.loc[panel.country.eq(c),v].notna().sum()); br.append(x)
breaks=pd.DataFrame(br); display(breaks.sort_values('p_bonf').round(4))

## 4. Two-way fixed effects and dependence diagnostics

Country/year fixed effects with one-year lagged predictors are associations, not causal effects. Only nine country clusters makes cluster-robust p-values approximate. Cross-sectional dependence below is a diagnostic approximation; validate with a dedicated econometric implementation before publication.

In [ ]:
for v in ['fdi_gdp','gfcf_gdp','exports_gdp','imports_gdp','hightech_exports_share','manufacturing_gdp']:
 panel[v+'_lag1']=panel.groupby('iso3')[v].shift(1)
models=[]
for y,xs in [('manufacturing_gdp',['fdi_gdp_lag1','gfcf_gdp_lag1','exports_gdp_lag1']),('exports_gdp',['manufacturing_gdp_lag1','fdi_gdp_lag1','gfcf_gdp_lag1']),('gfcf_gdp',['fdi_gdp_lag1','manufacturing_gdp_lag1'])]:
 z=panel[[y]+xs+['iso3','year']].dropna()
 if len(z)>50 and z.iso3.nunique()>=5:
  m=smf.ols(f'{y} ~ '+' + '.join(xs)+' + C(iso3) + C(year)',data=z).fit(cov_type='cluster',cov_kwds={'groups':z.iso3})
  for x in xs: models.append({'outcome':y,'predictor':x,'coef':m.params[x],'cluster_se':m.bse[x],'p_raw':m.pvalues[x],'n':int(m.nobs),'countries':z.iso3.nunique()})
reg=pd.DataFrame(models)
if len(reg): reg['p_BH']=multipletests(reg.p_raw,method='fdr_bh')[1]
display(reg.round(4))
def cd_test(v):
 w=panel.pivot(index='year',columns='country',values=v); w=w.loc[w.notna().sum(axis=1)>=5]; w=w.loc[:,w.notna().sum(axis=0)>=10]
 if w.shape[0]<10 or w.shape[1]<5:return {'variable':v,'N':w.shape[1],'T':w.shape[0],'CD':np.nan,'p':np.nan}
 c=w.corr(min_periods=10); vals=c.where(np.triu(np.ones(c.shape),1).astype(bool)).stack().dropna(); N,T=w.shape
 cd=np.sqrt(2*T/(N*(N-1)))*vals.sum()
 return {'variable':v,'N':N,'T':T,'CD':cd,'p':2*stats.norm.sf(abs(cd)),'mean_pair_corr':vals.mean()}
cd=pd.DataFrame([cd_test(v) for v in ['manufacturing_gdp','exports_gdp','imports_gdp','fdi_gdp','gfcf_gdp','manufacturing_growth','gdp_growth']]); cd['p_BH']=multipletests(cd.p.fillna(1),method='fdr_bh')[1]; display(cd.round(4))

## 5. ADF/KPSS and lead-lag checks

ADF's null is a unit root; KPSS's null is stationarity. Disagreement is informative. Lead-lag rank correlations are not evidence of a literal baton pass and are corrected for multiple testing.

In [ ]:
st=[]
for c in countries:
 for v in ['manufacturing_gdp','exports_gdp','imports_gdp','fdi_gdp','gfcf_gdp','manufacturing_growth']:
  z=panel.loc[panel.country.eq(c),['year',v]].dropna().sort_values('year')[v]
  if len(z)<15: continue
  for kind,s in [('level',z),('first_difference',z.diff().dropna())]:
   if len(s)<12: continue
   try: ap=adfuller(s,autolag='AIC',regression='ct' if kind=='level' else 'c')[1]
   except Exception: ap=np.nan
   try: kp=kpss(s,regression='ct' if kind=='level' else 'c',nlags='auto')[1]
   except Exception: kp=np.nan
   st.append({'country':c,'variable':v,'series':kind,'n':len(s),'ADF_p':ap,'KPSS_p':kp})
stationarity=pd.DataFrame(st); display(stationarity.round(4))
ll=[]
for v in ['manufacturing_gdp','exports_gdp','hightech_exports_share','gfcf_gdp']:
 for peer in ['China','Korea, Rep.','Viet Nam']:
  for lag in [0,5,10]:
   for mode in ['levels','annual_change']:
    a=panel[panel.country.eq('Bangladesh')].set_index('year')[v]; b=panel[panel.country.eq(peer)].set_index('year')[v]
    if mode=='annual_change': a=a.diff(); b=b.diff()
    z=pd.concat([a.rename('bd'),b.rename('peer')],axis=1); z['peer_lag']=z.peer.shift(lag); z=z[['bd','peer_lag']].dropna()
    if len(z)>=12:
     rho,p=stats.spearmanr(z.bd,z.peer_lag); ll.append({'variable':v,'peer':peer,'lag':lag,'mode':mode,'n':len(z),'rho':rho,'p_raw':p})
leadlag=pd.DataFrame(ll)
if len(leadlag): leadlag['p_BH']=multipletests(leadlag.p_raw,method='fdr_bh')[1]
display(leadlag.sort_values('p_BH').round(4))

## Interpretation and next data needs

- Industrial structure: manufacturing share alone is insufficient; pair it with real manufacturing growth and subsector data.
- Trade: exports/GDP measures integration, not upgrading. Use **UN Comtrade or CEPII BACI** HS product-level trade to calculate revealed comparative advantage (RCA), export diversification and product overlap with earlier geese.
- Investment: GFCF/GDP is total fixed investment; FDI/GDP is volatile and does not prove technology transfer.
- Candidate breaks need adequate sample sizes, adjusted significance and specification stability.
- Unless robust product-level upgrading and changing comparative advantage are shown, the defensible conclusion is **partially consistent with, but insufficient to establish, the Flying Geese hypothesis**.

Suggested additions: UN Comtrade/CEPII BACI; UNIDO INDSTAT subsector value added/employment; WDI labor productivity, industrial employment, R&D and patents. Archive raw API data and retrieval date.